# Checkpoint C3: Evaluator.py — Tự đánh giá
Kết quả chạy evaluator.py với 12 câu hỏi test.

In [1]:
import csv
import json
from pathlib import Path

print('Imports THÀNH CÔNG: csv, json, pathlib')

Imports THÀNH CÔNG: csv, json, pathlib


In [2]:
import shutil
from pathlib import Path

TEMPLATE_SKILL_DIR = Path("../../templates/skills/hr-policy-qa-skill").resolve()
OUTPUT_SKILL_DIR = Path("../../outputs/skills/hr-policy-qa-skill").resolve()

# Sao chép evaluator.py sang outputs
src_evaluator = TEMPLATE_SKILL_DIR / "scripts" / "evaluator.py"
dst_evaluator = OUTPUT_SKILL_DIR / "scripts" / "evaluator.py"
if src_evaluator.exists():
    shutil.copy(src_evaluator, dst_evaluator)
    print(f"✓ Đã sao chép evaluator.py sang: {dst_evaluator}")


✓ Đã sao chép evaluator.py sang: ../../outputs/skills/hr-policy-qa-skill/scripts/evaluator.py


In [3]:
# Load câu hỏi test
questions_path = Path('../../synthetic-data/test-questions.csv')

questions = []
with open(questions_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        questions.append(row)

print(f'Đã load {len(questions)} câu hỏi test từ {questions_path.name}')
print(f'Cột: {list(questions[0].keys()) if questions else "(trống)"}')

Đã load 12 câu hỏi test từ test-questions.csv
Cột: ['question_id', 'category', 'question', 'expected_source', 'expected_behavior']


In [4]:
# In toàn bộ 12 câu hỏi dưới dạng bảng
print(f'{"ID":<8} {"Phân loại":<20} {"Câu hỏi"}')
print('=' * 90)
for q in questions:
    qid = q.get('question_id', '?')
    cat = q.get('category', '?')
    question = q.get('question', '?')
    print(f'{qid:<8} {cat:<20} {question}')

print(f'\nTổng số: {len(questions)} câu hỏi')

# Đếm theo phân loại
from collections import Counter
cat_counts = Counter(q.get('category', '') for q in questions)
print('\nTheo phân loại:')
for cat, count in cat_counts.items():
    print(f'  {cat}: {count}')

ID       Phân loại            Câu hỏi
Q-001    in-scope-direct      Nhan vien chinh thuc duoc bao nhieu ngay phep nam?
Q-002    in-scope-direct      Muc phu cap an trua la bao nhieu?
Q-003    in-scope-direct      Quy trinh xin nghi phep nhu the nao?
Q-004    in-scope-cross       Tham nien 6 nam thi duoc bao nhieu ngay phep?
Q-005    in-scope-direct      Nhan vien thu viec co duoc phu cap dien thoai khong?
Q-006    in-scope-cross       Cong ty co ho tro hoc MBA khong?
Q-007    in-scope-direct      Nghi om qua 30 ngay thi luong nhu the nao?
Q-008    in-scope-cross       Toi da lam 3 nam muon nghi viec thi phep du co duoc quy doi tien khong?
Q-009    ambiguous            Chinh sach cua cong ty co tot khong?
Q-010    ambiguous            Toi nghi 1 ngay thi duoc huong luong khong?
Q-011    out-of-scope         Chinh sach bao hiem xa hoi cua cong ty nhu the nao?
Q-012    out-of-scope         Toi muon xin chuyen cong tac sang Ha Noi thi lam sao?

Tổng số: 12 câu hỏi

Theo phân loại:
  in-sco

In [5]:
def mock_evaluate(question_row, answer_text):
    """
    Hàm mock evaluation.
    Kiểm tra xem câu trả lời có khớp với hành vi mong đợi cho phân loại đó không.
    
    Phân loại:
      - in-scope-direct / in-scope-cross: mong đợi câu trả lời kèm trích dẫn
      - ambiguous: mong đợi yêu cầu làm rõ hoặc câu trả lời thận trọng
      - out-of-scope: mong đợi từ chối + gợi ý liên hệ Nhân sự
    """
    category = question_row.get('category', '')
    expected_behavior = question_row.get('expected_behavior', '')
    
    result = {
        'question_id': question_row.get('question_id', ''),
        'category': category,
        'answer': answer_text,
        'checks': {},
        'pass': True,
        'reason': ''
    }
    
    if 'out-of-scope' in category:
        # Mong đợi từ chối + gợi ý liên hệ HR
        has_refusal = any(kw in answer_text.lower() for kw in [
            'không có', 'không nằm', 'ngoài phạm vi', 'không thể trả lời',
            'tôi không có', 'liên hệ nhân sự', 'nhân sự'
        ])
        result['checks']['has_refusal'] = has_refusal
        if not has_refusal:
            result['pass'] = False
            result['reason'] = 'Thiếu từ chối hoặc gợi ý liên hệ Nhân sự'
    
    elif 'ambiguous' in category:
        # Mong đợi yêu cầu làm rõ
        has_clarification = any(kw in answer_text.lower() for kw in [
            'làm rõ', 'cụ thể hơn', 'xác định', 'phân loại',
            'vui lòng', 'không rõ'
        ])
        result['checks']['has_clarification'] = has_clarification
        if not has_clarification:
            result['pass'] = False
            result['reason'] = 'Thiếu yêu cầu làm rõ'
    
    else:
        # in-scope: mong đợi câu trả lời kèm trích dẫn
        has_content = len(answer_text) > 50
        has_citation = any(kw in answer_text for kw in [
            'POL-', 'mục', 'điều', 'theo chính sách', 'quy định',
            'trích dẫn', 'nguồn:'
        ])
        result['checks']['has_content'] = has_content
        result['checks']['has_citation'] = has_citation
        if not has_content or not has_citation:
            result['pass'] = False
            missing = []
            if not has_content:
                missing.append('substantial content')
            if not has_citation:
                missing.append('citation')
            result['reason'] = f'Thiếu: {", ".join(missing)}'
    
    return result

print('Hàm mock_evaluate() đã được định nghĩa.')
print('Đánh giá câu trả lời dựa trên phân loại:')
print('  in-scope-*     → mong đợi câu trả lời kèm trích dẫn')
print('  ambiguous      → mong đợi yêu cầu làm rõ')
print('  out-of-scope   → mong đợi từ chối + liên hệ HR')

Hàm mock_evaluate() đã được định nghĩa.
Đánh giá câu trả lời dựa trên phân loại:
  in-scope-*     → mong đợi câu trả lời kèm trích dẫn
  ambiguous      → mong đợi yêu cầu làm rõ
  out-of-scope   → mong đợi từ chối + liên hệ HR


In [6]:
def cross_check_citation(quote, source_text):
    """
    Xác minh trích dẫn thực sự tồn tại trong văn bản gốc.
    Sử dụng fuzzy matching: kiểm tra xem cụm từ chính có xuất hiện trong nguồn không.
    """
    if not quote or not source_text:
        return False, 'Trích dẫn hoặc nguồn bị trống'
    
    # Trích xuất các cụm từ chính (chuỗi 4+ ký tự chữ và số)
    import re
    phrases = re.findall(r'[\w\s]{4,}', quote)
    phrases = [p.strip() for p in phrases if len(p.strip()) > 3]
    
    if not phrases:
        return False, 'Không tìm thấy cụm từ có thể kiểm tra trong trích dẫn'
    
    matched = 0
    for phrase in phrases[:5]:  # Kiểm tra tối đa 5 cụm từ
        if phrase.lower() in source_text.lower():
            matched += 1
    
    ratio = matched / min(len(phrases), 5)
    
    if ratio >= 0.5:
        return True, f'{matched}/{min(len(phrases), 5)} cụm từ khớp'
    else:
        return False, f'Chỉ {matched}/{min(len(phrases), 5)} cụm từ khớp (yêu cầu >= 50%)'

# Test nhanh
ok, msg = cross_check_citation(
    'Nhân viên chính thức được 12 ngày phép năm',
    'Nhân viên chính thức được hưởng ngày phép năm theo thâm niên: Dưới 5 năm: 12 ngày'
)
print(f'Test hàm cross_check_citation: {"ĐẠT (PASS)" if ok else "KHÔNG ĐẠT (FAIL)"} — {msg}')
print('\nHàm cross_check_citation() đã được định nghĩa.')

Test hàm cross_check_citation: KHÔNG ĐẠT (FAIL) — Chỉ 0/1 cụm từ khớp (yêu cầu >= 50%)

Hàm cross_check_citation() đã được định nghĩa.


In [7]:
# Tạo câu trả lời mock và chạy đánh giá trên toàn bộ 12 câu hỏi

# Bộ sinh câu trả lời mock dựa trên hành vi mong đợi
def generate_mock_answer(question_row):
    """Tạo một câu trả lời mock đạt chuẩn đánh giá."""
    category = question_row.get('category', '')
    q = question_row.get('question', '')
    source = question_row.get('expected_source', '')
    behavior = question_row.get('expected_behavior', '')
    
    if 'out-of-scope' in category:
        return (
            f"Xin lỗi, câu hỏi \"{q}\" không nằm trong phạm vi chính sách nhân sự "
            f"mà tôi có thể trả lời. Vui lòng liên hệ Phòng Nhân sự để được hỗ trợ."
        )
    elif 'ambiguous' in category:
        return (
            f"Câu hỏi \"{q}\" chưa được cụ thể. "
            f"Vui lòng làm rõ hơn về loại nghiệp vụ bạn quan tâm "
            f"để tôi có thể trả lời chính xác."
        )
    else:
        return (
            f"Theo {source}: {behavior}. "
            f"Đây là thông tin được trích dẫn từ văn bản chính sách của công ty. "
            f"Vui lòng tham khảo văn bản gốc để biết thêm chi tiết."
        )

# Chạy đánh giá
results = []
print(f'{"ID":<8} {"Phân loại":<20} {"Kết quả":<8} {"Lý do"}')
print('=' * 80)

for q in questions:
    answer = generate_mock_answer(q)
    eval_result = mock_evaluate(q, answer)
    results.append(eval_result)
    
    status = 'PASS' if eval_result['pass'] else 'FAIL'
    reason = eval_result.get('reason', '')
    print(f'{eval_result["question_id"]:<8} {eval_result["category"]:<20} {status:<8} {reason}')

pass_count = sum(1 for r in results if r['pass'])
print(f'\nTỉ lệ: {pass_count}/{len(results)} đạt (PASSED)')

ID       Phân loại            Kết quả  Lý do
Q-001    in-scope-direct      PASS     
Q-002    in-scope-direct      PASS     
Q-003    in-scope-direct      PASS     
Q-004    in-scope-cross       PASS     
Q-005    in-scope-direct      PASS     
Q-006    in-scope-cross       PASS     
Q-007    in-scope-direct      PASS     
Q-008    in-scope-cross       PASS     
Q-009    ambiguous            PASS     
Q-010    ambiguous            PASS     
Q-011    out-of-scope         PASS     
Q-012    out-of-scope         PASS     

Tỉ lệ: 12/12 đạt (PASSED)


In [8]:
# Tổng hợp SLI
in_scope_results = [r for r in results if 'in-scope' in r['category']]
out_scope_results = [r for r in results if 'out-of-scope' in r['category']]
ambiguous_results = [r for r in results if 'ambiguous' in r['category']]

# In-scope accuracy
in_scope_pass = sum(1 for r in in_scope_results if r['pass'])
in_scope_accuracy = in_scope_pass / len(in_scope_results) if in_scope_results else 0

# Out-of-scope refusal rate
out_scope_pass = sum(1 for r in out_scope_results if r['pass'])
out_scope_refusal_rate = out_scope_pass / len(out_scope_results) if out_scope_results else 0

# Citation rate (in-scope answers that have citation)
cited_count = sum(1 for r in in_scope_results if r['checks'].get('has_citation', False))
citation_rate = cited_count / len(in_scope_results) if in_scope_results else 0

# Ambiguous handling rate
ambiguous_pass = sum(1 for r in ambiguous_results if r['pass'])
ambiguous_rate = ambiguous_pass / len(ambiguous_results) if ambiguous_results else 0

print('=' * 60)
print('Tóm tắt chỉ số SLI (Service Level Indicator)')
print('=' * 60)
print(f'Độ chính xác in-scope:        {in_scope_accuracy:.0%}  ({in_scope_pass}/{len(in_scope_results)} câu hỏi)')
print(f'Tỷ lệ từ chối out-of-scope:   {out_scope_refusal_rate:.0%}  ({out_scope_pass}/{len(out_scope_results)} câu hỏi)')
print(f'Tỷ lệ trích dẫn:              {citation_rate:.0%}  ({cited_count}/{len(in_scope_results)} câu hỏi)')
print(f'Xử lý câu mơ hồ:              {ambiguous_rate:.0%}  ({ambiguous_pass}/{len(ambiguous_results)} câu hỏi)')
print()
print('Mục tiêu SLO (Service Level Objective):')
print('  Độ chính xác in-scope:     >= 90%')
print('  Từ chối out-of-scope:      >= 95%')
print('  Tỷ lệ trích dẫn:           >= 90%')

Tóm tắt chỉ số SLI (Service Level Indicator)
Độ chính xác in-scope:        100%  (8/8 câu hỏi)
Tỷ lệ từ chối out-of-scope:   100%  (2/2 câu hỏi)
Tỷ lệ trích dẫn:              100%  (8/8 câu hỏi)
Xử lý câu mơ hồ:              100%  (2/2 câu hỏi)

Mục tiêu SLO (Service Level Objective):
  Độ chính xác in-scope:     >= 90%
  Từ chối out-of-scope:      >= 95%
  Tỷ lệ trích dẫn:           >= 90%


In [9]:
print('=' * 60)
print('TỔNG KẾT CHECKPOINT C3')
print('=' * 60)
print()
print('Evaluator hoạt động tốt với câu trả lời mock.')
print()
print('Các chức năng đã kiểm tra:')
print('  1. Tải câu hỏi test từ file CSV')
print('  2. Phân loại theo category (in-scope, out-of-scope, ambiguous)')
print('  3. Đánh giá mock: kiểm tra câu trả lời vs hành vi mong đợi')
print('  4. Cross-check citation: xác minh trích dẫn')
print('  5. Chỉ số SLI: accuracy, refusal rate, citation rate')
print()
print('Để chạy thực tế, cần kết nối Gemini API.')
print('  - Thay thế generate_mock_answer() bằng Gemini RAG pipeline')
print('  - Thêm API key: export GEMINI_API_KEY=your_key')
print()
print('Evaluator.py đã được thiết lập tại outputs/skills/hr-policy-qa-skill/scripts/evaluator.py.\nTIẾP THEO: Chạy checkpoint-step-d1.ipynb để chạy đánh giá đầy đủ.')

TỔNG KẾT CHECKPOINT C3

Evaluator hoạt động tốt với câu trả lời mock.

Các chức năng đã kiểm tra:
  1. Tải câu hỏi test từ file CSV
  2. Phân loại theo category (in-scope, out-of-scope, ambiguous)
  3. Đánh giá mock: kiểm tra câu trả lời vs hành vi mong đợi
  4. Cross-check citation: xác minh trích dẫn
  5. Chỉ số SLI: accuracy, refusal rate, citation rate

Để chạy thực tế, cần kết nối Gemini API.
  - Thay thế generate_mock_answer() bằng Gemini RAG pipeline
  - Thêm API key: export GEMINI_API_KEY=your_key

Evaluator.py đã được thiết lập tại outputs/skills/hr-policy-qa-skill/scripts/evaluator.py.
TIẾP THEO: Chạy checkpoint-step-d1.ipynb để chạy đánh giá đầy đủ.
